In [3]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/KDDCup09_upselling.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
pd.set_option("display.max_columns", None)
df.head()

,Var6,Var13,Var21,Var22,Var24,Var25,Var28,Var35,Var38,Var57,Var65,Var73,Var74,Var76,Var78,Var81,Var83,Var85,Var109,Var112,Var113,Var119,Var123,Var125,Var126,Var132,Var133,Var134,Var140,Var144,Var149,Var153,Var160,Var163,Var196,Var203,Var205,Var207,Var208,Var210,Var211,Var218,Var221,Var223,Var227,UPSELLING
0,9723.0,908.0,4568.0,5710.0,54.0,840.0,287.76,0.0,3712122.0,2.320231,9.0,32.0,105.0,2663904.0,0.0,16084.53,590.0,102.0,1232.0,1736.0,67110.00,11350.0,1608.0,12465.0,-20.0,0.0,3666460.0,901128.0,5065.0,0.0,1140734.0,10151040.0,766.0,695202.0,0,0,1,10,0,5,1,cJvF,4,0,2,-1
1,840.0,3108.0,212.0,265.0,6.0,88.0,166.56,0.0,15900.0,1.325785,27.0,126.0,343.0,324960.0,0.0,14406.96,0.0,8.0,24.0,72.0,10296.44,690.0,6.0,92097.0,32.0,0.0,543125.0,160770.0,3110.0,18.0,230930.0,1064396.0,56.0,202914.0,0,0,1,9,0,5,0,UYBR,6,0,1,-1
2,259.0,0.0,128.0,160.0,6.0,128.0,200.00,0.0,0.0,6.782952,9.0,18.0,0.0,795360.0,0.0,30113.10,0.0,4.0,56.0,72.0,4948.44,460.0,0.0,0.0,4.0,0.0,270145.0,280598.0,0.0,9.0,201691.0,1221824.0,10.0,70248.0,0,0,1,10,0,5,0,cJvF,4,0,2,-1
3,630.0,596.0,24.0,30.0,0.0,0.0,186.64,0.0,5485038.0,0.371502,9.0,100.0,231.0,2779944.0,0.0,194665.20,10.0,0.0,24.0,8.0,-64855.20,230.0,18.0,28665.0,-30.0,0.0,0.0,0.0,415.0,27.0,0.0,9885080.0,0.0,0.0,0,0,2,10,0,5,0,UYBR,4,3,2,-1
4,140.0,0.0,64.0,80.0,0.0,24.0,220.08,0.0,4279404.0,3.092074,9.0,16.0,0.0,0.0,0.0,124983.90,5.0,4.0,0.0,0.0,451612.00,460.0,18.0,0.0,-30.0,0.0,5866200.0,804074.0,0.0,9.0,0.0,9355360.0,14.0,273504.0,0,0,1,10,0,5,0,UYBR,4,0,2,-1


In [5]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [7]:
column_types = {
        "sufficient_resources_nominal_columns" : ['Var196', 'Var203', 'Var205', 'Var207', 'Var208', 'Var210', 'Var211',
                    'Var218', 'Var221', 'Var223', 'Var227'],
        "insufficient_resources_nominal_columns": None,
        "binary": None,
        "ordinal": None,
        "inclusion": None,
        "similarity": None,
    }
ANOC_threshold = 100

In [9]:
df['UPSELLING'] = df['UPSELLING'].astype(int)
df['UPSELLING'] = df['UPSELLING'].replace(-1,0)
y = df["UPSELLING"]
X = df.drop(["UPSELLING"], axis=1)

In [11]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.base import clone
from meta_encoder import MetaEncoder

# models
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#from approach import encode_with_approach

model_types = {
    "LogReg": "ATI",
    "MLP": "ATI",
    "LinearSVM": "ATI",
    "CART": "tree",
    "RF": "tree",
    "CatBoost": "tree",
    "XGBoost": "tree",
    "LightGBM": "tree"
}
target_encoders = ['Leave-one-out', 'K-fold', 'CatBoost', 'CV-GLMM', 'WOE']

nominal_cols = column_types["sufficient_resources_nominal_columns"].copy()
if column_types["insufficient_resources_nominal_columns"]:
    nominal_cols.extend(column_types["insufficient_resources_nominal_columns"])

max_categories = df[nominal_cols].nunique().max()
ANOC = df.shape[0] / max_categories
# ANOC = 101

encode_with_target: bool = False

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():

    # initializing guide parameters
    if model_types[model_name] == 'ATI':
        if ANOC > ANOC_threshold:
            encode_with_target = False
        else:
            encode_with_target = True
    else:  # 'tree-based'
        if ANOC > ANOC_threshold:
            encode_with_target = True
        else:
            encode_with_target = False

    target_encoders_to_apply = []

    if encode_with_target:
        target_encoders_to_apply = target_encoders
    else: # it just a mark that we use target-agnostic encoders
        target_encoders_to_apply = ["OneHot"]

    for nominal_encoder in target_encoders_to_apply:

        # creating parameters for the meta-encoder
        encode_columns_as = {}
        for type, columns in column_types.items():
            if type == "binary":
                encode_columns_as["Binary"] = columns
                continue
            if type == "similarity":
                encode_columns_as["Similarity"] = columns
                continue
            if type == "ordinal":
                encode_columns_as["Ordinal"] = columns
                continue
            if type == "inclusion":
                encode_columns_as["RankHot"] = columns
                continue

        encoder_name: str
        if not encode_with_target:
            if column_types["sufficient_resources_nominal_columns"] :
                encode_columns_as["OneHot"] = column_types["sufficient_resources_nominal_columns"]
                encoder_name = "OneHot"
            if column_types["insufficient_resources_nominal_columns"] :
                encode_columns_as["Hash"] = column_types["insufficient_resources_nominal_columns"]
                if encoder_name != '':
                    encoder_name = encoder_name + "_Hash"
        else:
            encode_columns_as[nominal_encoder] = nominal_cols
            encoder_name = nominal_encoder

        encoder = MetaEncoder(encode_columns_as=encode_columns_as)

        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            X_train_encoded = encoder.fit_transform(X_train, y_train)
            X_test_encoded = encoder.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Approach",
                    "dataset": "KDDCup09",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Approach, KDDCup09, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_KDDCup09_approach.csv", index=True)
print("Done!")

Approach, KDDCup09, LogReg, OneHot, 1, 1, 0.8288088926756918, 0.8301606387657857, 0.7269116186693148, 0.7301275760549558, 0.7209302325581395, 0.7395626242544732, 0.19197916984558105
Approach, KDDCup09, LogReg, OneHot, 1, 2, 0.8530799330998138, 0.854264027958925, 0.76266137040715, 0.7659157688540646, 0.7562862669245648, 0.7757936507936508, 0.13403987884521484
Approach, KDDCup09, LogReg, OneHot, 1, 3, 0.8346224837851618, 0.8339968106346602, 0.7495029821073559, 0.7567567567567568, 0.7354596622889306, 0.7793240556660039, 0.13672518730163574
Approach, KDDCup09, LogReg, OneHot, 1, 4, 0.8306226260725904, 0.8306119955037137, 0.7495029821073559, 0.7514792899408284, 0.7455968688845401, 0.757455268389662, 0.13368439674377441
Approach, KDDCup09, LogReg, OneHot, 1, 5, 0.8463809587801225, 0.8488896445952088, 0.7564612326043738, 0.757185332011893, 0.7549407114624506, 0.7594433399602386, 0.13524746894836426
Approach, KDDCup09, LogReg, OneHot, 2, 1, 0.8525119126510775, 0.8528731010759851, 0.75670307845